# DeFi exploration — live + backfilled history

Reads `data/defi.duckdb`. Live snapshot tables (`hl_ctx`, `cex_spot`, `pm_book`) plus backfilled history (`hl_candles`, `cex_candles`, `hl_funding_hist`) from `backfill_history.py` — so lead-lag and basis have real numbers now.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
print('connected:', ROOT/'data'/'defi.duckdb')

## Coverage (live snapshots + backfilled history)

In [ ]:
for t in ['hl_ctx','hl_book','cex_spot','pm_book','hl_candles','cex_candles','hl_funding_hist','pm_price_hist','pm_resolved']:
    try:
        col = 't' if t.endswith(('candles','hist')) or t=='hl_funding_hist' else ('captured_at' if t!='pm_resolved' else 'end_date')
        n,mn,mx = con.execute(f'select count(*), min({col}), max({col}) from {t}').fetchone()
        print(f'{t:16} rows={n:>7}  {mn} -> {mx}')
    except Exception as e:
        print(f'{t:16} -- {str(e)[:50]}')

## Hyperliquid funding & open interest (latest snapshot, ~230 coins)

In [ ]:
ctx = con.execute('''WITH l AS (SELECT max(captured_at) m FROM hl_ctx)
  SELECT coin, funding, open_interest FROM hl_ctx, l WHERE captured_at=l.m''').df()
top = ctx.sort_values('open_interest', ascending=False).head(20).copy()
top['funding_apr'] = top['funding']*24*365*100
fig, ax = plt.subplots(1,2, figsize=(13,6))
sns.barplot(data=top, y='coin', x='open_interest', ax=ax[0], color='steelblue'); ax[0].set_title('Open interest (top 20)')
sns.barplot(data=top.sort_values('funding_apr'), y='coin', x='funding_apr', ax=ax[1], color='indianred'); ax[1].set_title('Funding (annualized %)')
plt.tight_layout(); plt.show()

## Funding rate over time (backfilled)

In [ ]:
fh = con.execute("SELECT coin, t, funding_rate*24*365*100 AS apr FROM hl_funding_hist WHERE coin IN ('BTC','ETH','SOL') ORDER BY t").df()
if len(fh):
    fig, ax = plt.subplots(figsize=(11,4))
    for c,g in fh.groupby('coin'): ax.plot(g.t, g.apr, label=c)
    ax.axhline(0,color='k',lw=.6); ax.set_ylabel('funding APR %'); ax.legend(); ax.set_title('Hyperliquid funding over time'); plt.show()
else: print('no funding history yet — run backfill_history.py')

## Basis over time: Hyperliquid mark vs Coinbase spot (1-min candles)

In [ ]:
basis = con.execute('''SELECT h.coin AS asset, h.t, h.close AS hl, c.close AS cx, h.close-c.close AS basis
  FROM hl_candles h JOIN cex_candles c ON h.coin=c.asset AND h.t=c.t
  WHERE h.interval='1m' ORDER BY h.t''').df()
fig, ax = plt.subplots(figsize=(12,5))
for a in ['BTC','ETH','SOL']:
    d = basis[basis.asset==a]
    if len(d): ax.plot(d.t, d.basis, label=f'{a} (n={len(d)})', lw=.8)
ax.axhline(0,color='k',lw=.6); ax.set_ylabel('HL mark - CEX (close)'); ax.legend(); ax.set_title('Basis over time'); plt.show()
basis.groupby('asset').basis.describe()[['count','mean','std','min','max']]

## CEX↔DEX lead-lag (1-min cross-correlation)

Correlate HL log-returns vs CEX log-returns at lag *k* minutes. Peak at +k ⇒ Hyperliquid **leads** (CEX follows k min later); at −k ⇒ CEX leads.

In [ ]:
def leadlag(asset, max_lag=5):
    d = basis[basis.asset==asset].sort_values('t')
    rh = np.diff(np.log(d.hl.to_numpy())); rc = np.diff(np.log(d.cx.to_numpy())); n=len(rh)
    lags, cors = list(range(-max_lag, max_lag+1)), []
    for L in lags:
        a,b = (rh[:n-L], rc[L:]) if L>=0 else (rh[-L:], rc[:n+L])
        cors.append(float(np.corrcoef(a,b)[0,1]) if len(a)>30 and a.std()>0 and b.std()>0 else np.nan)
    return lags, cors
fig, ax = plt.subplots(figsize=(9,5))
for a in ['BTC','ETH','SOL']:
    lags, cors = leadlag(a)
    if np.all(np.isnan(cors)): continue
    ax.plot(lags, cors, marker='o', label=a)
    k = int(np.nanargmax(cors)); best=lags[k]
    who = 'HL leads' if best>0 else ('CEX leads' if best<0 else 'sync')
    print(f'{a}: peak lag {best:+d} min (corr {cors[k]:+.3f}) -> {who}')
ax.axvline(0,color='k',lw=.6); ax.set_xlabel('lag (min); +=HL leads'); ax.set_ylabel('return corr'); ax.legend(); ax.set_title('Lead-lag'); plt.show()

## Baseline predictive check: does the basis predict the next-minute CEX move?

Sign agreement between basis(t) and the next-minute CEX return. >50% would be a (weak) signal.

In [ ]:
rows=[]
for a in basis.asset.unique():
    d = basis[basis.asset==a].sort_values('t')
    if len(d)<100: continue
    b = d.basis.to_numpy()[:-1]; fwd = np.diff(d.cx.to_numpy())
    m = fwd!=0
    rows.append((a, int(m.sum()), round(float(np.mean(np.sign(b[m])==np.sign(fwd[m]))),3)))
pd.DataFrame(rows, columns=['asset','n','basis_predicts_next_move_hitrate'])